<a href="https://colab.research.google.com/github/PIER-202621036/proyecto-gestion-vacaciones/blob/main/notebooks/03_criticidad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **INICIO**

In [ ]:
import pandas as pd
import numpy as np

from IPython.display import display
from datetime import date

# 1. Configurar Pandas para que NO parta la tabla ni oculte columnas en la consola
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)



# ==================================================================
# 1.-ZONA COMUN: Cargar excel (datos.py)
# ==================================================================

URL_EXCEL = (
    "https://raw.githubusercontent.com/"
    "PIER-202621036/proyecto-gestion-vacaciones/"
    "main/data/empleados_vacaciones.xlsx"
)

# Hojas que usa Python (las demas del Excel son material de gestion)
EXCEL_HOJAS_PRINCIPALES = [
    "Empleados",
    "Puestos",
    "Vacaciones",
    "Programacion_Vacaciones",
    "Ofertas_Compensacion",
    "Parametros_Vacaciones",
    "Parametros_Criticidad",
    "Niveles_Criticidad"
]

# Unicas columnas que pueden estar vacias (segun el Diccionario)
NULOS_PERMITIDOS = ["motivo", "fecha_estado"]

negocio_excel = {}
negocio_parametros = {}

# Cargar todas las hojas del Excel
"""Lee el Excel y devuelve los datos ORIGINALES, sin limpiar.
    El resultado es un diccionario {nombre_de_hoja: lista de diccionarios}.
    Cada fila del Excel es un diccionario; la celda vacia queda como None.
"""
try:
  excel_hojas = pd.read_excel(URL_EXCEL, sheet_name=None)
except OSError as error:
  raise ConnectionError(f"No se pudo leer el Excel: {error}") from error


for nombre_hoja in EXCEL_HOJAS_PRINCIPALES:
  if nombre_hoja not in excel_hojas:
      raise ValueError(f"El Excel no tiene la hoja {nombre_hoja}")

  df = excel_hojas[nombre_hoja]

  # Reemplaza donde pd.isna() sea True *****one
  negocio_excel[nombre_hoja] = df.mask(pd.isna(df), None)

# Mostrar las hojas
print("HOJAS PRINCIPALES DEL EXCEL")
print("=" * 60)
for nombre_hoja in negocio_excel:
    print("*", nombre_hoja)

print("\n\nMOSTRANDO EMPLEADOS")
print("=" * 60)
display(negocio_excel["Empleados"])


HOJAS PRINCIPALES DEL EXCEL
* Empleados
* Puestos
* Vacaciones
* Programacion_Vacaciones
* Ofertas_Compensacion
* Parametros_Vacaciones
* Parametros_Criticidad
* Niveles_Criticidad


MOSTRANDO EMPLEADOS


,id,nombre,puesto_id,salario,fecha_ingreso
0,1,Verónica Ramírez,P18,3450,2026-07-20
1,2,Ricardo Ramírez,P11,4550,2023-08-26
2,3,Karen López,P01,4600,2022-08-07
3,4,Elena Quispe,P19,5850,2025-08-31
4,5,Renzo Vega,P13,14600,2018-01-25
...,...,...,...,...,...
995,996,Milagros Núñez,P16,8050,2021-08-01
996,997,María Lazo,P11,5100,2023-03-14
997,998,Daniela Salazar,P07,18450,2020-04-30
998,999,Ricardo Ramos,P09,11050,2021-01-24


In [ ]:
# ==================================================================
# 3.-ZONA COMUN: Limpieza y validación (datos.py)
# ==================================================================

# Iteramos hoja por hoja
for nombre_hoja, df in negocio_excel.items():

    # 1. Duplicados
    cantidad_duplicados = df.duplicated().sum()
    if cantidad_duplicados > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_duplicados} duplicados.")
        df = df.drop_duplicates()

    # 2. Nulos (excluyendo NULOS_PERMITIDOS)
    columnas_obligatorias = [
        col
        for col in df.columns
        if col.lower() not in [n.lower() for n in NULOS_PERMITIDOS]
    ]
    cantidad_nulos = df[columnas_obligatorias].isna().sum().sum()
    if cantidad_nulos > 0:
        print(f"Hoja {nombre_hoja}: {cantidad_nulos} nulos no permitidos.")
        df = df.dropna(subset=columnas_obligatorias)

    # Guardamos la hoja limpia de vuelta en el diccionario
    negocio_excel[nombre_hoja] = df.reset_index(drop=True)

### **3.CRITICIDAD**



In [ ]:
# Buscar un puesto por su id y mostrar su detalle (incluye las 4 notas de criticidad)
puesto_id = "P09"

df_puestos = negocio_excel["Puestos"]
df_puesto = df_puestos[df_puestos["puesto_id"] == puesto_id]   # filtrado condicional

print(f"\n\nMostrando detalles del Puesto: {puesto_id}")
print("="*140)
if df_puesto.empty:
    print(f"No existe el puesto {puesto_id}")
else:
    display(df_puesto)



Mostrando detalles del Puesto: P09


,puesto_id,puesto,nivel,area,banda_min,banda_max,banda_salarial,importancia,dificultad_reemplazo,impacto_ausencia,disponibilidad_reemplazo
8,P09,Senior Geologist,Senior,Geología,10000,12000,"S/ 10,000 - 12,000",5,4,5,3


In [ ]:
# Cada variable se califica del 1 al 5,
# donde el 1 es lo menos grave y el 5 tiene mayor impacto.

df_reglas = negocio_excel["Parametros_Criticidad"]

print("\n\nReglas de criticidad (cada variable se califica con una nota):")
print("="*140)
display(df_reglas[["variable", "peso", "nota_min", "nota_max", "nota_1_significa", "nota_5_significa"]])



Reglas de criticidad (cada variable se califica con una nota):


,variable,peso,nota_min,nota_max,nota_1_significa,nota_5_significa
0,importancia,0.40,1,5,"Puesto de apoyo, bajo aporte estratégico",Puesto estratégico para el negocio
1,dificultad_reemplazo,0.25,1,5,Se reemplaza en pocos días,"Perfil escaso, reemplazarlo toma meses"
2,impacto_ausencia,0.25,1,5,La operación casi no se afecta,Su ausencia detiene o pone en riesgo la operación
3,disponibilidad_reemplazo,0.10,1,5,Varios compañeros pueden cubrirlo de inmediato,Nadie puede cubrirlo internamente


In [ ]:
# Cálculo del score
# Para P09 la tabla debe dar 40, 20, 25 y 6 puntos, que suman 91.
# Cada fila sale de (nota / 5 × peso × 100);
# por ejemplo, la de dificultad_reemplazo es 4 / 5 × 0.25 × 100 = 20.

puesto = df_puesto.to_dict(orient="records")[0]      # el puesto como diccionario
reglas = df_reglas.to_dict(orient="records")         # cada regla como diccionario

filas_calculo = []
score = 0

for regla in reglas:
    nombre = regla["variable"]
    nota = puesto[nombre]                            # nota del puesto en esta variable

    if nota < regla["nota_min"] or nota > regla["nota_max"]:
        print(f"ATENCION: la nota de {nombre} ({nota}) esta fuera de {regla['nota_min']} a {regla['nota_max']}")

    puntos = nota / regla["nota_max"] * regla["peso"] * 100
    score += puntos

    filas_calculo.append({
        "variable": nombre,
        "nota": nota,
        "nota_max": regla["nota_max"],
        "peso": regla["peso"],
        "puntos": round(puntos, 2),
    })

score = round(score, 2)

print(f"\n\nCalculo del score del Puesto: {puesto_id}")
print("="*140)
display(pd.DataFrame(filas_calculo))
print(f"SCORE = {score}")



Calculo del score del Puesto: P09


,variable,nota,nota_max,peso,puntos
0,importancia,5,5,0.40,40.0
1,dificultad_reemplazo,4,5,0.25,20.0
2,impacto_ausencia,5,5,0.25,25.0
3,disponibilidad_reemplazo,3,5,0.10,6.0


SCORE = 91.0


In [14]:
#El resultado es Score 91.0 -> Nivel Critico

df_niveles = excel_hojas["Niveles_Criticidad"]
niveles = df_niveles.to_dict(orient="records")
niveles.sort(key=lambda nivel: nivel["score_min"], reverse=True)   # del nivel mas alto al mas bajo

nivel_puesto = None
for nivel in niveles:
    if score >= nivel["score_min"]:
        nivel_puesto = nivel["nivel"]
        break

print(f"\n\nNivel de criticidad del Puesto: {puesto_id}")
print("="*140)
display(df_niveles)
print(f"Score {score} -> Nivel {nivel_puesto}")


######################
#OTROS CASOS
######################

# Para probar otro puesto,
# se cambia solo el puesto_id en la celda 1
# y se vuelve a correr las cuatra celdas.
# puesto_id = P12 sale 20.0 y Bajo,
# y con puesto_id = P03 sale 100 y Critico.



Nivel de criticidad del Puesto: P09


,score_min,score_max,nivel
0,0,39.99,Bajo
1,40,69.99,Medio
2,70,84.99,Alto
3,85,100.00,Critico


Score 91.0 -> Nivel Critico
